### Master Seminar Physical Geography 2026, University of Graz
--------
## Elevational Temperature Gradient and seasonal variability 
## in Greenland 2010-2026
#### Lea Pietrek & Miriam Erdler (2026)
---
#### 💡 Aim of the Project: characterize how elevational temperature gradients across Greenland have evolved using PROMICE Station Pairs


---
---

## Settings

In [ ]:
# install necessary libraries
!pip install pandas numpy matplotlib seaborn scipy pymannkendall jupyter

In [ ]:
# import necessary libraries
import pandas as pd #reading and manipulating data
import numpy as np #numerical operations
from pathlib import Path #handling file paths
import pymannkendall as mk #Mann-Kendall trend test
import matplotlib.patches as mpatches #for legend
import matplotlib.pyplot as plt #plotting
from scipy import stats #for linear regression
import matplotlib.lines as mlines #for legend

------
------

## Part A: Data Cleaning

#### Defining constants

Paths: This section defines all folder paths used in the workflow. Using relative paths instead of local paths ensures that the notebook can be executed on different computers without modifying the code

In [ ]:
# Define paths 

# project root 
PROJECT_ROOT = Path.cwd()

# paths
RAW_DATA = PROJECT_ROOT / "Data" / "Data_RAW"

PROCESSED_DATA = PROJECT_ROOT / "Data" / "Data_Processed"

CLEAN_DATA = PROCESSED_DATA / "Cleaned_Stations"

FIGURE_DIR     = PROJECT_ROOT / "Figures"

RESULTS_DIR    = PROJECT_ROOT / "Results"

# create folders if they don't exist yet 
CLEAN_DATA.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Define constants for seasons
SEASONS = ["DJF", "MAM", "JJA", "SON"]

# Map month to season
SEASON_MAP = {
    1:"DJF", 2:"DJF", 3:"MAM", 4:"MAM",  5:"MAM",
    6:"JJA", 7:"JJA", 8:"JJA", 9:"SON", 10:"SON",
    11:"SON", 12:"DJF"
}

In [ ]:
# Function to convert p-value to significance stars
def stars(p):
    """Return significance stars for a p-value."""
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "n.s."

Let's insepct the data! This code will check for the existing CSV Files.

In [ ]:
# List all CSV files in the raw data directory
# The workflow is designed to automatically detect all PROMICE station files, 
# making it scalable and reducing the need for manual file selection.

csv_files = list(RAW_DATA.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files")

for file in csv_files:
    print(file.name)

Let's continue inspecting the data and have a closer look at the dataframe

In [ ]:
# Read a sample file to check the structure

sample_file = csv_files[0]

df = pd.read_csv(sample_file)

df.head()

In [ ]:
# Check column names
df.columns

##### Extract Metadata from File Names:
PROMICE file names contain important metadata information.
Example:

`KAN_L_month.csv`

From this naming structure we extract:
- station ID (`KAN`)
- station level (`L` = lower station, `U` = upper station)
- temporal resolution (`month`)
  
This metadata will later help identify station pairs and distinguish between elevation levels.

This function extracts these three pieces of information automatically from any filename,
so the code works for all 18 files without manual input.

In [ ]:
# Function to extract station information from file name
def extract_station_info(file_path):
    """
    KAN_L_month.csv → ("KAN", "L", "month")
    """
    parts = file_path.stem.split("_")
    return parts[0], parts[1], parts[2]

### Define Cleaning Function
This function applies all cleaning steps to a single PROMICE station file.

**Steps performed:**
1. Extract station metadata from the filename
2. Load the raw CSV file
3. Replace PROMICE no-data codes (`-999`, `-9999`) with `NaN`
4. Keep only the columns relevant for this analysis
5. Parse the `time` column as a proper datetime format
6. Filter rows to the study period 2010–2026
7. Replace physically impossible temperatures with `NaN`
   (values outside –70°C to +30°C cannot occur in Greenland)
8. Remove duplicate rows and sort chronologically
9. Add metadata columns (`station_id`, `site`, `position`, `resolution`)
10. Add a season label to each row (DJF / MAM / JJA / SON)

In [ ]:
# Function to clean one PROMICE file
def clean_promice_file(file_path):
    """
    Load and clean one raw PROMICE CSV.
    Always reads from RAW_DATA — never modifies the original file.
    """
    station_id, station_level, temporal_resolution = extract_station_info(file_path)

    # Load raw file
    df = pd.read_csv(file_path)

    # Replace PROMICE no-data codes with NaN
    df = df.replace([-999, -9999], np.nan)

    # Keep only columns needed for analysis
    columns_to_keep = ["time", "gps_lat", "gps_lon", "gps_alt", "t_u", "rh_u"]
    df = df[[c for c in columns_to_keep if c in df.columns]]

    # Parse time as datetime
    df["time"] = pd.to_datetime(df["time"])

    # Filter to study period 2010–2026
    df = df[df["time"].dt.year.between(2010, 2026)]

    # Remove physically impossible temperatures
    if "t_u" in df.columns:
        df.loc[~df["t_u"].between(-70, 30), "t_u"] = np.nan

    # Remove duplicates, sort by time
    df = df.drop_duplicates().sort_values("time").reset_index(drop=True)

    # Add metadata from filename
    df["station_id"] = f"{station_id}_{station_level}"
    df["site"]       = station_id
    df["position"]   = station_level
    df["resolution"] = temporal_resolution
    df["season"]     = df["time"].dt.month.map(SEASON_MAP)



    return df

### Clean and Save All 18 Station Files
The cleaning function is applied to every raw file automatically.

Each cleaned file is saved individually to `Data_Processed/Cleaned_Stations/`
using the same naming structure as the raw file, with `_clean` added.

Example: `KAN_L_month.csv` → `KAN_L_month_clean.csv`

In [ ]:
# Apply cleaning to every raw file and save to Cleaned_Stations 
# The output file name is derived from the input file name, with a "_clean" suffix for clarity.
for file in csv_files:

    df = clean_promice_file(file)

    station_id, station_level, temporal_resolution = extract_station_info(file)

    out_name = f"{station_id}_{station_level}_{temporal_resolution}_clean.csv"
    df.to_csv(CLEAN_DATA / out_name, index=False)

    print(f"  {file.name:25s} → {len(df):3d} rows | t_u NaN: {df['t_u'].isna().sum()}")

##### Data Quality Check

In [ ]:
# Show % missing per variable per station
missing = {}

for file in sorted(CLEAN_DATA.glob("*.csv")):
    df = pd.read_csv(file)
    missing[file.stem] = df.isna().mean() * 100

pd.DataFrame(missing).round(1)

In [ ]:
# Load all cleaned files 
all_stations = pd.concat(
    [pd.read_csv(f, parse_dates=["time"]) for f in sorted(CLEAN_DATA.glob("*.csv"))],
    ignore_index=True
)

Fixed elevation per station: The GPS value is not always the same resulting in noisy values for the analysis; furthermore, sometimes GPS values are missing in some months leading to NA, even tho temperature has been measured. 

To fix this issue, we applied a fixed mean elevation per station instead of teh row-by-row GPS value. 

In [ ]:
# Compute fixed elevation per station 
# gps_alt is recorded row-by-row but can be noisy or missing (e.g. SCO)
# using the median across all available readings gives a stable reference elevation
elev_upper = (all_stations[all_stations["position"] == "U"]
              .groupby("site")["gps_alt"]
              .median()
              .rename("elev_upper_fixed"))

elev_lower = (all_stations[all_stations["position"] == "L"]
              .groupby("site")["gps_alt"]
              .median()
              .rename("elev_lower_fixed"))

print("\nFixed elevations per site (m a.s.l.):")
print(pd.concat([elev_upper, elev_lower], axis=1).round(1))

### Build Stations Pairs + compute Temperature Gradient

In [ ]:
# Build station pairs 
upper = all_stations[all_stations["position"] == "U"]
lower = all_stations[all_stations["position"] == "L"]

pairs = pd.merge(
    upper, lower,
    on=["site", "time", "season", "resolution"],
    suffixes=("_upper", "_lower")
)

# Merge fixed elevations and compute gradient 
# fixed elevations replace noisy row-by-row GPS for the gradient calculation
pairs = pairs.merge(elev_upper, on="site")
pairs = pairs.merge(elev_lower, on="site")

pairs["elev_diff_m"] = pairs["elev_upper_fixed"] - pairs["elev_lower_fixed"]

# temperature gradient in °C per 100m — main analysis variable
pairs["gradient_per100m"] = (
    (pairs["t_u_upper"] - pairs["t_u_lower"]) / pairs["elev_diff_m"]
) * 100

# drop only rows where temperature is NaN (not where GPS was NaN)
pairs = pairs.dropna(subset=["gradient_per100m"])

print(f"\n✅ {len(pairs)} paired records | {pairs['site'].nunique()} station pairs")
print(pairs.groupby("site")["gradient_per100m"].agg(["count", "mean"]).round(3))


Save the final station pairs

In [ ]:
# Save — this is the only file the analysis section needs
out_path = PROCESSED_DATA / "station_pairs_monthly.csv"
pairs.to_csv(out_path, index=False)
print(f"\n✅ Saved: {out_path.name}")

------
------

## Part B: Core Analysis 

#### 💡 This Part of the Code focuses on the main analysis of the temperature data 

### Trend Analysis

#### Mann-Kendall Test and Sen's Slope for all months

In [ ]:
results = []

for site, grp in pairs.groupby("site"):
    # sort chronologically and drop missing gradient values
    series = grp.sort_values("time")["gradient_per100m"].dropna()

    # run Mann-Kendall
    res = mk.original_test(series)

    results.append({
        "site"           : site,
        "n"              : len(series),
        "trend"          : res.trend,       # "increasing" / "decreasing" / "no trend"
        "tau"            : round(res.Tau, 3),
        "p_value"        : round(res.p, 4),
        "sen_slope_month": round(res.slope, 6),        # °C/100m per month
        "sen_slope_year" : round(res.slope * 12, 5),   # easier to interpret
    })

# Results table
mk_df = pd.DataFrame(results)
print(mk_df.to_string(index=False))

------

### Seasonality

#### Mann-Kendall Test and Sen's Slope for seasons

In [ ]:
results = []

# Seasonal analysis: Mann-Kendall test for each site and season
for site, grp in pairs.groupby("site"):
    for season in SEASONS:
        # filter to this season and drop NaN
        series = (grp[grp["season"] == season]
                  .sort_values("time")["gradient_per100m"]
                  .dropna())

        # skip if too few observations (< 10 = unreliable)
        if len(series) < 10:
            print(f"  {site} {season}: skipped (n={len(series)})")
            continue

        res = mk.original_test(series)

        results.append({
            "site"           : site,
            "season"         : season,
            "n"              : len(series),
            "trend"          : res.trend,
            "tau"            : round(res.Tau, 3),
            "p_value"        : round(res.p, 4),
            "sen_slope_month": round(res.slope, 6),
            "sen_slope_year" : round(res.slope * 12, 5),
        })

seasonal_df = pd.DataFrame(results)


In [ ]:
## Clean seasonal summary table with explicit MK and Sen Slope labels
# Build a multi-level summary table 

seasonal_df["sig"] = seasonal_df["p_value"].apply(stars) 

rows = []

for _, row in seasonal_df.iterrows():
    rows.append({
        "site"   : row["site"],
        "season" : row["season"],
        "MK trend"     : row["trend"],           # Mann-Kendall result
        "MK p-value"   : row["p_value"],         # Mann-Kendall p-value
        "MK sig"       : row["sig"],             # significance stars
        "Sen slope\n(°C/100m/yr)": row["sen_slope_year"],  # Sen's Slope rate
    })

detail_df = pd.DataFrame(rows)

# Print per season for clarity 
for season in SEASONS:
    print(f"\n{'─'*60}")
    print(f"Season: {season}")
    print(f"{'─'*60}")
    sub = detail_df[detail_df["season"] == season].drop(columns="season")
    print(sub.to_string(index=False))

print("\nMann-Kendall: tests WHETHER a trend exists (p-value + direction)")
print("Sen's Slope:  measures HOW FAST it changes (°C/100m/year)")

Plotting the Resulsts for Seasonality based on MK-Test and Sen's Slope

In [ ]:
## Heatmap: Seasonal Sen's Slope with poster color scheme

# Imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

PROJECT_ROOT = Path.cwd()
FIGURE_DIR   = PROJECT_ROOT / "Figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SEASONS = ["DJF", "MAM", "JJA", "SON"]

# Significance stars helper
def stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "n.s."

# Poster colormap
poster_cmap = LinearSegmentedColormap.from_list(
    "poster_diverging",
    [(0.0,   "#485E86"),
     (0.35,  "#A4B3CF"),
     (0.5,   "#FFFFFF"),
     (0.65,  "#FFD96A"),
     (0.8,   "#FAB400"),
     (1.0,   "#C45C00")]
)

# Pivot Sen's slope and p-values to wide format 
slope_pivot = seasonal_df.pivot(index="site", columns="season",
                                values="sen_slope_year")[SEASONS]
pval_pivot  = seasonal_df.pivot(index="site", columns="season",
                                values="p_value")[SEASONS]

# Plot
fig, ax = plt.subplots(figsize=(8, 6))

vmax = np.abs(slope_pivot.values).max()
im   = ax.imshow(slope_pivot.values, cmap=poster_cmap,
                 vmin=-vmax, vmax=vmax, aspect="auto")

# Axis labels
ax.set_xticks(range(len(SEASONS)))
ax.set_xticklabels(SEASONS)
ax.set_yticks(range(len(slope_pivot.index)))
ax.set_yticklabels(slope_pivot.index)
ax.set_xlabel("Season")
ax.set_ylabel("Station pair")

# Title and significance note
ax.set_title("Saisonality of Temperature Gradients")

# small italic note bottom right
fig.text(0.02, 0.01,
         "Mann-Kendall significance: \n * p<0.05   ** p<0.01   *** p<0.001",
         ha="left", va="bottom", fontsize=8, color="#555555", style="italic")

# Annotate each cell with slope value + significance stars
for i, site in enumerate(slope_pivot.index):
    for j, season in enumerate(SEASONS):
        slope = slope_pivot.loc[site, season]
        p     = pval_pivot.loc[site, season]
        sig   = stars(p)

        # white text on dark cells, black text on light cells
        text_color = "white" if abs(slope) > vmax * 0.5 else "black"

        ax.text(j, i, f"{slope:.3f}\n{sig}",
                ha="center", va="center",
                fontsize=8, color=text_color, linespacing=1.5)

# Colorbar
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("°C/100m/year")

fig.tight_layout()
fig.subplots_adjust(bottom=0.1)
fig.savefig(FIGURE_DIR / "seasonal_senslope_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: seasonal_senslope_heatmap.png")

------

### Seasonality: Do DJF and JJA significantly differ from each other​?

##### Wilcoxon-Signed Rank Test for Winter and Summer

In [ ]:
pairs = pd.read_csv(PROCESSED_DATA / "station_pairs_monthly.csv", parse_dates=["time"])

results = []

# Seasonal comparison: Wilcoxon signed-rank test between DJF and JJA gradients for each site
for site, grp in pairs.groupby("site"):
    djf = grp[grp["season"] == "DJF"]["gradient_per100m"].dropna().values
    jja = grp[grp["season"] == "JJA"]["gradient_per100m"].dropna().values

    # Wilcoxon signed-rank requires equal length paired samples
    # use the shorter length to pair them
    n = min(len(djf), len(jja))

# Skip if too few paired observations
    if n < 5:
        print(f"  {site}: skipped (n={n})")
        continue

# Perform Wilcoxon signed-rank test on the paired samples
    stat, p = stats.wilcoxon(djf[:n], jja[:n])

# Store results with medians and significance
    results.append({
        "site"       : site,
        "n_paired"   : n,
        "median_DJF" : round(np.median(djf), 4),
        "median_JJA" : round(np.median(jja), 4),
        "difference" : round(np.median(djf) - np.median(jja), 4),
        "W_stat"     : round(stat, 2),
        "p_value"    : round(p, 4),
        "sig"        : stars(p),
    })

wilcoxon_df = pd.DataFrame(results)

# Print results
print("Wilcoxon Signed-Rank Test: DJF vs JJA gradient")
print("Negative difference = DJF gradient more negative than JJA = steeper in winter")
print()
print(wilcoxon_df.to_string(index=False))

# Save

wilcoxon_df.to_csv(RESULTS_DIR / "wilcoxon_djf_jja.csv", index=False)
print("✅ Saved: Results/wilcoxon_djf_jja.csv")

Plotting the resulst from the Wilcoxon signed rank test for seasonality

In [ ]:
PROJECT_ROOT   = Path.cwd()
PROCESSED_DATA = PROJECT_ROOT / "Data" / "Data_Processed"
RESULTS_DIR    = PROJECT_ROOT / "Results"
FIGURE_DIR     = PROJECT_ROOT / "Figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Load data 
pairs        = pd.read_csv(PROCESSED_DATA / "station_pairs_monthly.csv", parse_dates=["time"])
wilcoxon_df  = pd.read_csv(RESULTS_DIR / "wilcoxon_djf_jja.csv")
# exclude TAS — insufficient paired observations (n=6)
wilcoxon_df = wilcoxon_df[wilcoxon_df["site"] != "TAS"].reset_index(drop=True)
# Colors
COLOR_DJF = "#485E86"   # blue = winter
COLOR_JJA = "#FAB400"   # orange = summer
COLOR_SIG = "black"     # significant connecting line
COLOR_NS  = "lightgrey" # non-significant connecting line

# Sort stations by DJF median for clean ordering
wilcoxon_df = wilcoxon_df.sort_values("median_DJF", ascending=True).reset_index(drop=True)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))

for i, row in wilcoxon_df.iterrows():

    # line color based on significance
    line_color = COLOR_SIG if row["sig"] != "n.s." else COLOR_NS

    # connecting line between DJF and JJA dot
    ax.plot([row["median_DJF"], row["median_JJA"]],
            [i, i],
            color=line_color, linewidth=2, zorder=1)

    # DJF dot
    ax.scatter(row["median_DJF"], i,
               color=COLOR_DJF, s=80, zorder=3)

    # JJA dot
    ax.scatter(row["median_JJA"], i,
               color=COLOR_JJA, s=80, zorder=3)

    # significance stars to the right of the plot
    ax.text(-1.15, i, row["sig"],
            ha="center", va="center", fontsize=9)
    
# set x limits to give space for stars on the left
ax.set_xlim(-1.25, 0.1)

# Axis formatting 
ax.set_yticks(range(len(wilcoxon_df)))
ax.set_yticklabels(wilcoxon_df["site"])
ax.set_xlabel("Median gradient (°C / 100m)")
ax.set_title("Seasonal contrast in elevational temperature gradient\nDJF vs JJA 2010-2026")

# 0°C reference line
ax.axvline(0, color="grey", linewidth=0.7, linestyle="--", alpha=0.6)

# Legend
legend_djf = mlines.Line2D([], [], color=COLOR_DJF, marker="o",
                            linestyle="None", markersize=8, label="DJF (winter)")
legend_jja = mlines.Line2D([], [], color=COLOR_JJA, marker="o",
                            linestyle="None", markersize=8, label="JJA (summer)")
legend_sig = mlines.Line2D([], [], color=COLOR_SIG, linewidth=2,
                            label="significant (p<0.05)")
legend_ns  = mlines.Line2D([], [], color=COLOR_NS, linewidth=2,
                            label="not significant")

ax.legend(handles=[legend_djf, legend_jja, legend_sig, legend_ns],
          fontsize=8, loc="lower right",
          bbox_to_anchor=(0.92, 0.02))

# significance note in a box at the bottom
fig.text(0.02, 0.01,
         "* p<0.05   ** p<0.01   *** p<0.001   n.s. not significant",
         ha="left", va="bottom", fontsize=7, color="#555555", style="italic",
         bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none"))

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.tight_layout()
fig.subplots_adjust(bottom=0.08)  
fig.savefig(FIGURE_DIR / "dumbbell_djf_jja.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: dumbbell_djf_jja.png")

------